In [ ]:
from sklearn.ensemble import RandomForestRegressor
from math import sqrt
from sklearn.metrics import mean_squared_error
import gc, joblib, time

print("🌲 Training Random Forest Regressor...")

t0 = time.time()
# Conservative settings for Colab (tweak later if stable)
rf_reg = RandomForestRegressor(
    n_estimators=50,      # start small; can increase to 100–200 later
    max_depth=10,         # limits tree depth to control memory
    min_samples_leaf=5,   # prevents overfitting on small subgroups
    n_jobs=1,             # single thread (safe for Colab Free)
    random_state=42
)

rf_reg.fit(X_train_s, y_reg_train)
train_time = time.time() - t0

y_pred_rf_reg = rf_reg.predict(X_test_s)
rf_rmse = sqrt(mean_squared_error(y_reg_test, y_pred_rf_reg))
print(f"✅ RF Regressor RMSE: {rf_rmse:.6f}")
print(f"Training time: {train_time:.2f}s")

# Save and cleanup
joblib.dump(rf_reg, "/content/rf_regressor.joblib")
del rf_reg, y_pred_rf_reg
gc.collect()
print("✅ Model saved and memory cleared.")

In [ ]:
import joblib
from math import sqrt
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score

# Example: load rf_reg and compute RMSE precisely
m = joblib.load("/content/rf_regressor.joblib")
pred = m.predict(X_test_s)
print("RF reg RMSE:", sqrt(mean_squared_error(y_reg_test, pred)))
del m; gc.collect()

In [ ]:
# Cell: Train-evaluate table (re-uses X_train_s, X_test_s, y_reg_train, y_reg_test, y_clf_train, y_clf_test)
import joblib, gc
from math import sqrt
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score

# Load saved models if present else train quick
models = {}
try:
    models['linear'] = joblib.load("/content/linear_regression.joblib")
    models['lasso'] = joblib.load("/content/lasso_reg.joblib")
    models['logr'] = joblib.load("/content/logistic.joblib")
    models['rf_reg'] = joblib.load("/content/rf_regressor.joblib")
    models['rf_clf'] = joblib.load("/content/rf_classifier.joblib")
    print("Loaded saved models.")
except Exception as e:
    print("Some models not found, train them manually (use previous cells).", e)

results = {}
# Regression
y_lr = models['linear'].predict(X_test_s)
results['Linear_RMSE'] = sqrt(mean_squared_error(y_reg_test, y_lr))

y_lasso = models['lasso'].predict(X_test_s)
results['Lasso_RMSE'] = sqrt(mean_squared_error(y_reg_test, y_lasso))

y_rfreg = models['rf_reg'].predict(X_test_s)
results['RFReg_RMSE'] = sqrt(mean_squared_error(y_reg_test, y_rfreg))

# Classification
y_logr = models['logr'].predict(X_test_s)
results['Logistic_Acc'] = accuracy_score(y_clf_test, y_logr)
results['Logistic_F1']  = f1_score(y_clf_test, y_logr, zero_division=0)

y_rfclf = models['rf_clf'].predict(X_test_s)
results['RFClf_Acc'] = accuracy_score(y_clf_test, y_rfclf)
results['RFClf_F1']  = f1_score(y_clf_test, y_rfclf, zero_division=0)

import pandas as pd
pd.DataFrame([results])

In [ ]:
# === Plot model evaluation results (bar charts + table) ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib
from math import sqrt
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score

# --- Helper: compute metrics from saved models if results not available ---
def compute_metrics_from_models(X_test, y_reg, y_clf):
    computed = {}
    # Regression models
    try:
        m_lin = joblib.load("/content/linear_regression.joblib")
        y_pred = m_lin.predict(X_test)
        computed["Linear_RMSE"] = sqrt(mean_squared_error(y_reg, y_pred))
    except Exception as e:
        print("Linear load/score error:", e)

    try:
        m_lasso = joblib.load("/content/lasso_reg.joblib")
        y_pred = m_lasso.predict(X_test)
        computed["Lasso_RMSE"] = sqrt(mean_squared_error(y_reg, y_pred))
    except Exception as e:
        print("Lasso load/score error:", e)

    try:
        m_rfreg = joblib.load("/content/rf_regressor.joblib")
        y_pred = m_rfreg.predict(X_test)
        computed["RFReg_RMSE"] = sqrt(mean_squared_error(y_reg, y_pred))
    except Exception as e:
        print("RFReg load/score error:", e)

    # Classification models
    try:
        m_logr = joblib.load("/content/logistic.joblib")
        y_pred = m_logr.predict(X_test)
        computed["Logistic_Acc"] = accuracy_score(y_clf, y_pred)
        computed["Logistic_F1"]  = f1_score(y_clf, y_pred, zero_division=0)
    except Exception as e:
        print("Logistic load/score error:", e)

    try:
        m_rfclf = joblib.load("/content/rf_classifier.joblib")
        y_pred = m_rfclf.predict(X_test)
        computed["RFClf_Acc"] = accuracy_score(y_clf, y_pred)
        computed["RFClf_F1"]  = f1_score(y_clf, y_pred, zero_division=0)
    except Exception as e:
        print("RFClf load/score error:", e)

    return computed

# --- Build results DataFrame ---
# If results already exists from your notebook, use it. Otherwise compute from saved models.
if 'results' in globals() and isinstance(results, dict) and len(results) > 0:
    res_dict = results.copy()
else:
    print("results not found or empty — attempting to compute from saved models...")
    try:
        res_dict = compute_metrics_from_models(X_test_s, y_reg_test, y_clf_test)
    except Exception as e:
        raise RuntimeError("Missing test arrays X_test_s / y_reg_test / y_clf_test in workspace. Error: " + str(e))

# Normalize key names for plotting and create DataFrame
# Keep consistent column names
mapping = {
    "Linear_RMSE":"Linear (RMSE)",
    "Lasso_RMSE":"Lasso (RMSE)",
    "RFReg_RMSE":"RandomForest Reg (RMSE)",
    "Logistic_Acc":"Logistic (Acc)",
    "Logistic_F1":"Logistic (F1)",
    "RFClf_Acc":"RF Classifier (Acc)",
    "RFClf_F1":"RF Classifier (F1)",
    "RF_Reg_RMSE":"RandomForest Reg (RMSE)",    # in case other keys exist
    "RF_Acc":"RF Classifier (Acc)",
    "RF_F1":"RF Classifier (F1)"
}

# Remap and keep only present keys
res_renamed = {}
for k,v in res_dict.items():
    key = mapping.get(k, k)
    res_renamed[key] = res_dict[k]

# Convert to DataFrame for nicer display
# Separate regression and classification metrics
reg_keys = [k for k in res_renamed.keys() if "(RMSE)" in k or "RMSE" in k]
clf_keys = [k for k in res_renamed.keys() if "(Acc)" in k or "(F1)" in k or "Acc" in k or "F1" in k]

reg_df = pd.DataFrame({
    "Model": reg_keys,
    "RMSE": [res_renamed[k] for k in reg_keys]
}).sort_values("RMSE")

# For classification, build tidy DataFrame with Acc and F1 columns
# Try to pair models by name prefix
clf_rows = {}
for k in clf_keys:
    if "Acc" in k:
        base = k.replace(" (Acc)","")
        clf_rows.setdefault(base, {})["Acc"] = res_renamed[k]
    elif "F1" in k:
        base = k.replace(" (F1)","")
        clf_rows.setdefault(base, {})["F1"] = res_renamed[k]
    else:
        # fallback: split heuristics
        if "Acc" in k:
            base = k.split("Acc")[0].strip()
            clf_rows.setdefault(base, {})["Acc"] = res_renamed[k]
        elif "F1" in k:
            base = k.split("F1")[0].strip()
            clf_rows.setdefault(base, {})["F1"] = res_renamed[k]
        else:
            clf_rows.setdefault(k, {})["Acc_or_val"] = res_renamed[k]

clf_df = pd.DataFrame.from_dict(clf_rows, orient="index").reset_index().rename(columns={"index":"Model"})
# Ensure columns exist
for col in ["Acc","F1","Acc_or_val"]:
    if col not in clf_df.columns:
        clf_df[col] = np.nan

# Save combined CSV summary
summary_df = pd.concat([
    reg_df.set_index("Model"),
    clf_df.set_index("Model")
], axis=1)
summary_df.to_csv("/content/model_evaluation_summary.csv")
print("Saved: /content/model_evaluation_summary.csv")

# === Plotting ===
plt.rcParams.update({"figure.facecolor":"white"})

fig, axes = plt.subplots(1,2, figsize=(14,5), gridspec_kw={'width_ratios':[1,1]})

# Left: Regression RMSE bar (lower is better)
if not reg_df.empty:
    ax = axes[0]
    ax.barh(reg_df["Model"], reg_df["RMSE"], color="tab:blue")
    ax.set_xlabel("RMSE (lower is better)")
    ax.set_title("Regression Models — RMSE")
    for i,v in enumerate(reg_df["RMSE"]):
        ax.text(v, i, f" {v:.4f}", va='center', fontsize=9)
else:
    axes[0].text(0.5, 0.5, "No regression results found", ha='center', va='center')

# Right: Classification grouped bar (Accuracy & F1)
ax = axes[1]
if not clf_df.empty:
    ind = np.arange(len(clf_df))
    width = 0.35
    acc_vals = clf_df["Acc"].fillna(0).values
    f1_vals  = clf_df["F1"].fillna(0).values

    b1 = ax.bar(ind - width/2, acc_vals, width, label="Accuracy")
    b2 = ax.bar(ind + width/2, f1_vals, width, label="F1-score")
    ax.set_xticks(ind)
    ax.set_xticklabels(clf_df["Model"], rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score (0-1)")
    ax.set_title("Classification Models — Accuracy & F1")
    ax.legend()
    # annotate
    for rect in b1 + b2:
        h = rect.get_height()
        ax.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
else:
    axes[1].text(0.5, 0.5, "No classification results found", ha='center', va='center')

plt.tight_layout()
plt.savefig("/content/model_evaluation_plots.png", dpi=150)
plt.show()

print("Saved plot: /content/model_evaluation_plots.png")
print("\nSummary DataFrame (head):")
display(summary_df.head(20))

In [ ]:
# === Top-11 weighted pies: Low-Risk (inverse-vol) and High-Risk (mean-return) — top-11 only ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Prepare pivot (Date x Ticker) robustly ---
try:
    pivot_for_vol = pivot_cluster.copy()    # prefer pivot_cluster if available
except Exception:
    try:
        pivot_for_vol = pivot_train.copy()  # else pivot_train
    except Exception:
        pivot_for_vol = (
            train_df.groupby(["Date","Ticker"])["DailyReturn"]
            .mean()
            .reset_index()
            .pivot(index="Date", columns="Ticker", values="DailyReturn")
        ).fillna(0.0)

# --- compute volatility and mean return ---
vols_series = pivot_for_vol.std().rename("Volatility")
mean_ret_series = pivot_for_vol.mean().rename("MeanReturn")

# --- Build low-risk (inverse-vol) and high-risk (mean-return) weights ---
inv_vol = 1.0 / vols_series.replace(0, np.nan)          # prevent /0
w_lowrisk_all = inv_vol / inv_vol.sum()

# For high risk, use positive mean returns (clip negatives to zero)
w_highrisk_all = mean_ret_series.clip(lower=0.0)
if w_highrisk_all.sum() == 0:
    # fallback: use raw mean returns (can be negative but we'll take abs)
    w_highrisk_all = mean_ret_series.abs()
w_highrisk_all = w_highrisk_all / (w_highrisk_all.sum() if w_highrisk_all.sum() > 0 else 1.0)

# --- Select top-11 only (no "Others" slice) ---
k = 11
k = min(k, len(w_lowrisk_all))   # guard if fewer tickers available

low_top = w_lowrisk_all.nlargest(k)
high_top = w_highrisk_all.nlargest(k)

# --- Plot side-by-side pies (top-11 only) ---
plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
patches, texts, autotexts = plt.pie(
    low_top.values,
    labels=low_top.index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.8
)
plt.title("Low-Risk Portfolio (Top-11)")
for t in texts + autotexts:
    t.set_fontsize(8)

plt.subplot(1,2,2)
patches, texts, autotexts = plt.pie(
    high_top.values,
    labels=high_top.index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.8
)
plt.title("High-Risk Portfolio (Top-11)")
for t in texts + autotexts:
    t.set_fontsize(8)

plt.tight_layout()
plt.savefig("/content/fig_top11_low_high_risk_pie_top11_only.png", dpi=150)
plt.show()

print("Saved: /content/fig_top11_low_high_risk_pie_top11_only.png")
print("\nLow-risk top labels:", list(low_top.index))
print("High-risk top labels:", list(high_top.index))

In [ ]:
# ==========================================
# 📊 Final Project Summary — All Key Metrics
# ==========================================
import pandas as pd
from tabulate import tabulate

# --- ⿡ Model Performance Summary ---
model_summary = pd.DataFrame([{
    "Linear_RMSE": results.get("Linear_RMSE"),
    "Lasso_RMSE": results.get("Lasso_RMSE"),
    "Logistic_Acc": results.get("Logistic_Acc"),
    "Logistic_F1": results.get("Logistic_F1"),
    "RF_Reg_RMSE": results.get("RF_Reg_RMSE"),
    "RF_Acc": results.get("RF_Acc"),
    "RF_F1": results.get("RF_F1")
}])

print("\n===============================")
print("📈 MODEL PERFORMANCE SUMMARY")
print("===============================")
print(tabulate(model_summary, headers="keys", tablefmt="psql", showindex=False))


# --- ⿢ Portfolio Optimization Results ---
portfolio_summary = pd.DataFrame(
    [res_pred, res_hist],
    index=["Predicted_Portfolio", "Historical_Portfolio"]
)

print("\n===============================")
print("💼 PORTFOLIO PERFORMANCE SUMMARY")
print("===============================")
print(tabulate(portfolio_summary.round(4), headers="keys", tablefmt="psql"))


# --- ⿣ Key Clustering Insights ---
cluster_info = cluster_df["Cluster"].value_counts().sort_index()
print("\n===============================")
print("🔶 CLUSTERING & DIVERSIFICATION SUMMARY")
print("===============================")
print(f"Total tickers clustered: {len(cluster_df)}")
print(f"Number of clusters: {cluster_info.shape[0]}")
print("\nCluster sizes:")
print(cluster_info.to_string())

# --- ⿤ Sharpe Comparison ---
print("\n===============================")
print("⚖  SHARPE RATIO COMPARISON")
print("===============================")
print(f"Predicted Portfolio Sharpe : {res_pred['Sharpe']:.3f}")
print(f"Historical Portfolio Sharpe: {res_hist['Sharpe']:.3f}")

# --- ⿥ Summary Statement ---
print("\n===============================")
print("✅ FINAL SUMMARY")
print("===============================")
print(f"""
The project successfully implemented a Big Data–driven Portfolio Optimization pipeline:

• Trained classical models: Linear, Lasso, Logistic, and Random Forest.
• Evaluated models using RMSE, Accuracy, and F1-score.
• Predicted expected stock returns using RF Regressor.
• Applied Markowitz Modern Portfolio Theory to construct efficient portfolios.
• Compared ML-predicted vs Historical portfolios:
   → Predicted Cumulative Return: {res_pred['Cumulative Return']:.2%}
   → Historical Cumulative Return: {res_hist['Cumulative Return']:.2%}
   → Sharpe Ratios: {res_pred['Sharpe']:.2f} vs {res_hist['Sharpe']:.2f}
• Clustered 34 stocks into {cluster_info.shape[0]} risk–return groups using K-Means & PCA.
• Visualized sentiment–return relationships and efficient frontiers.
""")
print("🎯 Project complete — results are ready for report and presentation!")
